<a href="https://colab.research.google.com/github/AlvaroGomez23/DAW2023-2025/blob/main/M5075_NF3_3_DataFrames_API_Activitat_3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Activitat 3a: Dataframe Apache Spark


<hr>
  <li>Autor: Francesc Barragán
  <li>Institut Sa Palomera (Blanes)
  <li>Versió: 1.0
  <li>Darrera Actualització: 15.02.2025
  <li>Python version: 3.12
  <li>Jupyter Notebook
  <li>Spark 3.5.4
<hr>

En las següents activitats anem a familiaritzar-nos amb l'ús del API de Dataframes de Spark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, current_date, year, monotonically_increasing_id, mean, round, min
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
spark = SparkSession.builder.getOrCreate()

1. A partir del fitxer [nombres.json](https://www.sapalomera.cat/moodlecf/pluginfile.php/40106/mod_folder/content/0/nombres.json?forcedownload=1), crea un DataFrame i realitza les següents operacions:

In [ ]:
df = spark.read.json("nombres.json")

df.show()

+--------+----+------+
|  Ciudad|Edad|Nombre|
+--------+----+------+
|   Elche|  45| Aitor|
|Alicante|  14|Marina|
|   Elche|  19| Laura|
|    Aspe|  45| Sonia|
|   Elche|NULL| Pedro|
+--------+----+------+



a) Crea una nova columna (columna ``Mayor30``) que indiqui si la persona és major de 30 anys.

In [ ]:
df = df.withColumn("Mayor30", when(col("Edad") > 30, True).otherwise(False))

df.show()

+--------+----+------+-------+
|  Ciudad|Edad|Nombre|Mayor30|
+--------+----+------+-------+
|   Elche|  45| Aitor|   true|
|Alicante|  14|Marina|  false|
|   Elche|  19| Laura|  false|
|    Aspe|  45| Sonia|   true|
|   Elche|NULL| Pedro|  false|
+--------+----+------+-------+



b) Crea una nueva columna (columna ``FaltanJubilacion``) que calcule cuantos años le faltan para jubilarse (supongamos que se jubila a los 67 años)

In [ ]:
df = df.withColumn("FaltanJubilacion", 67 - col("Edad"))

df.show()

+--------+----+------+-------+----------------+
|  Ciudad|Edad|Nombre|Mayor30|FaltanJubilacion|
+--------+----+------+-------+----------------+
|   Elche|  45| Aitor|   true|              22|
|Alicante|  14|Marina|  false|              53|
|   Elche|  19| Laura|  false|              48|
|    Aspe|  45| Sonia|   true|              22|
|   Elche|NULL| Pedro|  false|            NULL|
+--------+----+------+-------+----------------+



c) Crea una nova columna (columna ``Apellidos``) que contingui ``XYZ`` (pots utilitzar la funció [lit](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lit.html))

In [ ]:
df = df.withColumn("Apellidos", lit("XYZ"))

df.show()

+--------+----+------+-------+----------------+---------+
|  Ciudad|Edad|Nombre|Mayor30|FaltanJubilacion|Apellidos|
+--------+----+------+-------+----------------+---------+
|   Elche|  45| Aitor|   true|              22|      XYZ|
|Alicante|  14|Marina|  false|              53|      XYZ|
|   Elche|  19| Laura|  false|              48|      XYZ|
|    Aspe|  45| Sonia|   true|              22|      XYZ|
|   Elche|NULL| Pedro|  false|            NULL|      XYZ|
+--------+----+------+-------+----------------+---------+



d) Elimina les columnes ``Mayor30`` y ``Apellidos``.

In [ ]:
df = df.drop("Mayor30", "Apellidos")

df.show()

+--------+----+------+----------------+
|  Ciudad|Edad|Nombre|FaltanJubilacion|
+--------+----+------+----------------+
|   Elche|  45| Aitor|              22|
|Alicante|  14|Marina|              53|
|   Elche|  19| Laura|              48|
|    Aspe|  45| Sonia|              22|
|   Elche|NULL| Pedro|            NULL|
+--------+----+------+----------------+



e) Crea una nova columna (columna ``AnyoNac``) amb l'any de naixement de cada persona (pots utilitzar la funció [current_date](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.current_date.html)).

In [ ]:
df = df.withColumn("AnyoNac", year(current_date()) - col("Edad"))
df.show()

+--------+----+------+----------------+-------+
|  Ciudad|Edad|Nombre|FaltanJubilacion|AnyoNac|
+--------+----+------+----------------+-------+
|   Elche|  45| Aitor|              22|   1981|
|Alicante|  14|Marina|              53|   2012|
|   Elche|  19| Laura|              48|   2007|
|    Aspe|  45| Sonia|              22|   1981|
|   Elche|NULL| Pedro|            NULL|   NULL|
+--------+----+------+----------------+-------+



d) Afegeix un id incremental per cada fila (camp ``Id``) i es que al fer un ``show`` es vegi en primer lloc (pots utilitzar la funció [monotonically_increasing_id](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.monotonically_increasing_id.html)) seguidos del ``Nombre``, ``Edad``, ``AnyoNac``, ``FaltaJubilacion`` y ``Ciudad``

In [ ]:
df = df.withColumn("Id", monotonically_increasing_id())

df = df.select(
    "Id",
    "Nombre",
    "Edad",
    "AnyoNac",
    "FaltanJubilacion",
    "Ciudad"
)

df.show()

+---+------+----+-------+----------------+--------+
| Id|Nombre|Edad|AnyoNac|FaltanJubilacion|  Ciudad|
+---+------+----+-------+----------------+--------+
|  0| Aitor|  45|   1981|              22|   Elche|
|  1|Marina|  14|   2012|              53|Alicante|
|  2| Laura|  19|   2007|              48|   Elche|
|  3| Sonia|  45|   1981|              22|    Aspe|
|  4| Pedro|NULL|   NULL|            NULL|   Elche|
+---+------+----+-------+----------------+--------+



Al realitzar els sis passos, el resultat del DataFrame serà semblant a:

```
+---+-------+----+-------+----------------+--------+
| Id|Nombre |Edad|AnyoNac|FaltanJubilacion|  Ciudad|
+---+-------+----+-------+----------------+--------+
|  0|  Aitor|  45|   1978|              22|   Elche|
|  1| Marina|  14|   2009|              53|Alicante|
|  2|  Laura|  19|   2004|              48|   Elche|
|  3|  Sonia|  45|   1978|              22|    Aspe|
|  4|  Pedro|null|   null|            null|   Elche|
+---+-------+----+-------+----------------+--------+
```

2. A partir del fitxer [VentasNulos.csv](https://www.sapalomera.cat/moodlecf/pluginfile.php/40106/mod_folder/content/0/VentasNulos.csv?forcedownload=1):

a) Elimina les files que tinguin al menys 4 nuls.

In [ ]:
df = spark.read.option("header", True).option("inferSchema", True).csv("VentasNulos.csv")

df.show()

+------+------+-----+-----------+-------------+
|Nombre|Ventas|Euros|     Ciudad|Identificador|
+------+------+-----+-----------+-------------+
|  Pepe|     4|  200|      Elche|          X21|
|Andreu|     8| NULL|       NULL|         NULL|
|  Juan|  NULL| NULL|       NULL|          C54|
| Pedro|     1|   30|   Valencia|          R23|
| María|  NULL|  300| Torrellano|         NULL|
|Marina|     3|  350|       Aspe|          V55|
|  NULL|    10|  500|Crevillente|          AMV|
|   Ana|    10| 2300|   Alicante|          B89|
|  NULL|  NULL| NULL|       NULL|         NULL|
| Jorge|     8| NULL|       NULL|          T19|
+------+------+-----+-----------+-------------+



In [ ]:
total_columns = len(df.columns)
min_non_nulls = total_columns - 3

df_clean = df.dropna(thresh=min_non_nulls)

df_clean.show()

+------+------+-----+-----------+-------------+
|Nombre|Ventas|Euros|     Ciudad|Identificador|
+------+------+-----+-----------+-------------+
|  Pepe|     4|  200|      Elche|          X21|
|Andreu|     8| NULL|       NULL|         NULL|
|  Juan|  NULL| NULL|       NULL|          C54|
| Pedro|     1|   30|   Valencia|          R23|
| María|  NULL|  300| Torrellano|         NULL|
|Marina|     3|  350|       Aspe|          V55|
|  NULL|    10|  500|Crevillente|          AMV|
|   Ana|    10| 2300|   Alicante|          B89|
| Jorge|     8| NULL|       NULL|          T19|
+------+------+-----+-----------+-------------+



b) Amb les files restants, substitueix:

i. Els noms nuls per ``Empleado``

In [ ]:
df_clean = df_clean.fillna({"Nombre": "Empleado"})

df_clean.show()

+--------+------+-----+-----------+-------------+
|  Nombre|Ventas|Euros|     Ciudad|Identificador|
+--------+------+-----+-----------+-------------+
|    Pepe|     4|  200|      Elche|          X21|
|  Andreu|     8| NULL|       NULL|         NULL|
|    Juan|  NULL| NULL|       NULL|          C54|
|   Pedro|     1|   30|   Valencia|          R23|
|   María|  NULL|  300| Torrellano|         NULL|
|  Marina|     3|  350|       Aspe|          V55|
|Empleado|    10|  500|Crevillente|          AMV|
|     Ana|    10| 2300|   Alicante|          B89|
|   Jorge|     8| NULL|       NULL|          T19|
+--------+------+-----+-----------+-------------+



ii. Les ventes nules per la mitja de les ventes dels companys (redondejant a enter).

In [ ]:
avg_ventas = df_clean.select(round(mean("Ventas")).alias("avg_ventas")).collect()[0]["avg_ventas"]

df_clean = df_clean.fillna({"Ventas": avg_ventas})

df_clean.show()

+--------+------+-----+-----------+-------------+
|  Nombre|Ventas|Euros|     Ciudad|Identificador|
+--------+------+-----+-----------+-------------+
|    Pepe|     4|  200|      Elche|          X21|
|  Andreu|     8| NULL|       NULL|         NULL|
|    Juan|     6| NULL|       NULL|          C54|
|   Pedro|     1|   30|   Valencia|          R23|
|   María|     6|  300| Torrellano|         NULL|
|  Marina|     3|  350|       Aspe|          V55|
|Empleado|    10|  500|Crevillente|          AMV|
|     Ana|    10| 2300|   Alicante|          B89|
|   Jorge|     8| NULL|       NULL|          T19|
+--------+------+-----+-----------+-------------+



### Agrupant
Per obtenir la mitjana, encara que ho veurem més endavant, has d'agrupar i després obtenir la mitjana de la columna:
```python
valor = df.groupBy().avg('Ventas')

```

iii. Els euros nuls pel valor del company que menys € ha guanyat. (després d'agrupar, pots utilitzar la funció ``min``)

In [ ]:
min_euros = df_clean.select(min("Euros")).collect()[0][0]

df_clean = df_clean.fillna({"Euros": min_euros})

df_clean.show()

+--------+------+-----+-----------+-------------+
|  Nombre|Ventas|Euros|     Ciudad|Identificador|
+--------+------+-----+-----------+-------------+
|    Pepe|     4|  200|      Elche|          X21|
|  Andreu|     8|   30|       NULL|         NULL|
|    Juan|     6|   30|       NULL|          C54|
|   Pedro|     1|   30|   Valencia|          R23|
|   María|     6|  300| Torrellano|         NULL|
|  Marina|     3|  350|       Aspe|          V55|
|Empleado|    10|  500|Crevillente|          AMV|
|     Ana|    10| 2300|   Alicante|          B89|
|   Jorge|     8|   30|       NULL|          T19|
+--------+------+-----+-----------+-------------+



iv. La ciutat nula per ``C.V.`` i el identificador nul per ``XYZ``


In [ ]:
df_clean = df_clean.fillna({
    "Ciudad": "C.V.",
    "Identificador": "XYZ"
})

df_clean.show()

+--------+------+-----+-----------+-------------+
|  Nombre|Ventas|Euros|     Ciudad|Identificador|
+--------+------+-----+-----------+-------------+
|    Pepe|     4|  200|      Elche|          X21|
|  Andreu|     8|   30|       C.V.|          XYZ|
|    Juan|     6|   30|       C.V.|          C54|
|   Pedro|     1|   30|   Valencia|          R23|
|   María|     6|  300| Torrellano|          XYZ|
|  Marina|     3|  350|       Aspe|          V55|
|Empleado|    10|  500|Crevillente|          AMV|
|     Ana|    10| 2300|   Alicante|          B89|
|   Jorge|     8|   30|       C.V.|          T19|
+--------+------+-----+-----------+-------------+



Pels passos ii) y iii) pots crear un DataFrame que obtingui el valor a assignar i després passar-lo com paràmetre al mètode per emplenar els nuls.

3. A partir del fitxxer [movies.tsv](https://www.sapalomera.cat/moodlecf/pluginfile.php/40106/mod_folder/content/0/movies.tsv?forcedownload=1), crea una esquema de forma declarativa amb els camps:

- ``interprete`` de tipo ``string``
- ``pelicula`` de tipo ``string``
- ``anyo`` de tipo ``int``

Cada fila del fitxer implica que el actor/actriu ha treballat en aquella pel·lícula en l'any indicat.
a) Una vegada creat l'esquema, carrega les dades en un DataFrame.

In [ ]:
esquema = StructType([
    StructField("interprete", StringType(), True),
    StructField("pelicula", StringType(), True),
    StructField("anyo", IntegerType(), True)
])

# Llegir el fitxer TSV amb l'esquema
df_movies = spark.read.csv(
    "movies.tsv",
    sep="\t",
    header=True,
    schema=esquema
)

# Mostrar les primeres files
df_movies.show()

+-----------------+--------------------+----+
|       interprete|            pelicula|anyo|
+-----------------+--------------------+----+
|McClure, Marc (I)|        Coach Carter|2005|
|McClure, Marc (I)|         Superman II|1980|
|McClure, Marc (I)|           Apollo 13|1995|
|McClure, Marc (I)|            Superman|1978|
|McClure, Marc (I)|  Back to the Future|1985|
|McClure, Marc (I)|Back to the Futur...|1990|
|Cooper, Chris (I)|  Me, Myself & Irene|2000|
|Cooper, Chris (I)|         October Sky|1999|
|Cooper, Chris (I)|              Capote|2005|
|Cooper, Chris (I)|The Bourne Supremacy|2004|
|Cooper, Chris (I)|         The Patriot|2000|
|Cooper, Chris (I)|            The Town|2010|
|Cooper, Chris (I)|          Seabiscuit|2003|
|Cooper, Chris (I)|      A Time to Kill|1996|
|Cooper, Chris (I)|Where the Wild Th...|2009|
|Cooper, Chris (I)|         The Muppets|2011|
|Cooper, Chris (I)|     American Beauty|1999|
|Cooper, Chris (I)|             Syriana|2005|
|Cooper, Chris (I)| The Horse Whis

A continuació, mitjançant el DataFrame API:

b) Mostra les pel·lícules en les que ha treballat ``Murphy, Eddie (I)``.

In [ ]:
df_movies.filter(col("interprete") == "Murphy, Eddie (I)").show()

+-----------------+--------------------+----+
|       interprete|            pelicula|anyo|
+-----------------+--------------------+----+
|Murphy, Eddie (I)|            Showtime|2002|
|Murphy, Eddie (I)|              Norbit|2007|
|Murphy, Eddie (I)|Hot Tub Time Machine|2010|
|Murphy, Eddie (I)|Nutty Professor I...|2000|
|Murphy, Eddie (I)|Beverly Hills Cop II|1987|
|Murphy, Eddie (I)|      Trading Places|1983|
|Murphy, Eddie (I)|      Daddy Day Care|2003|
|Murphy, Eddie (I)|      Dr. Dolittle 2|2001|
|Murphy, Eddie (I)| Shrek Forever After|2010|
|Murphy, Eddie (I)|   Beverly Hills Cop|1984|
|Murphy, Eddie (I)|               Shrek|2001|
|Murphy, Eddie (I)| The Haunted Mansion|2003|
|Murphy, Eddie (I)|   Coming to America|1988|
|Murphy, Eddie (I)|             Shrek 2|2004|
|Murphy, Eddie (I)|     Doctor Dolittle|1998|
|Murphy, Eddie (I)| The Nutty Professor|1996|
|Murphy, Eddie (I)|               Mulan|1998|
|Murphy, Eddie (I)|         Tower Heist|2011|
|Murphy, Eddie (I)|          Dream

c) Mostra els intèrprets que apareixen tant a ``Superman`` com a ``Superman II``.

In [ ]:
df_movies.filter((col("pelicula") == "Superman") | (col("pelicula") == "Superman II")).show()

+--------------------+-----------+----+
|          interprete|   pelicula|anyo|
+--------------------+-----------+----+
|   McClure, Marc (I)|Superman II|1980|
|   McClure, Marc (I)|   Superman|1978|
|    Lipinski, Eugene|Superman II|1980|
|       Calder, David|   Superman|1978|
|      Brando, Marlon|   Superman|1978|
|        Tuerpe, Paul|   Superman|1978|
|   Marzello, Vincent|   Superman|1978|
|Griffiths, Richar...|Superman II|1980|
|    Perrine, Valerie|   Superman|1978|
|    Perrine, Valerie|Superman II|1980|
|     Chancer, Norman|Superman II|1980|
|       Hackman, Gene|Superman II|1980|
|       Hackman, Gene|   Superman|1978|
|      Stamp, Terence|   Superman|1978|
|      Stamp, Terence|Superman II|1980|
|    Hollis, John (I)|   Superman|1978|
|    Hollis, John (I)|Superman II|1980|
|        Lyons, Derek|Superman II|1980|
|        Kahan, Steve|   Superman|1978|
|    Jurgensen, Randy|   Superman|1978|
+--------------------+-----------+----+
only showing top 20 rows
